# Lab 9 — TRIK gyroscope parameters (yaw axis)
**Charos · 240070 · CS413 Robotics**

Robot program: `charos_240070_lab9.qrs` (same structure as the course template `Gyro_demo`: thread **task** logs, thread **main** drives, **Kill Thread** ends logging).
Sequence: warm-up 2 min (not logged) → **static** 3 min → **Option A** rotation: **sin** 36 s, **rect** 36 s ($T = 4$ s, $A$ = 30 → 60 → 150 %, $u_R = u$, $u_L = -u$, clipped to ±100 %) → **rest** 3 s.
Log `lab9_gyro.csv`: `t_us, gz_cal, encL, encR, t_ms` (`t_us` = `read()[3]`, `gz_cal` = `read()[2]` in mdeg/s, encoders in degrees, `t_ms` = `time()`).

$$\omega_{meas}(t_k) = Q_{\Delta}\Big(\mathrm{sat}_{\pm\omega_{max}}\big[(1+s)\,\omega_{true}(t_k) + b_0 + d\,t_k + n_k\big]\Big), \quad n_k \sim \mathcal{N}(0,\sigma^2)$$

## 0. Settings and log

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
plt.rcParams.update({'figure.figsize': (11, 4), 'axes.grid': True})

LOG = 'lab9_gyro.csv'
r, L = 0.042, 0.194          # wheel radius and wheel base [m] (Lab 5 / course template, real robot)
SLIP_DEG = 0.0               # floor-mark slip check: angle left between nose and tape mark after the run [deg], + = CCW
WARMUP_S = 120               # schedule of the robot program (seconds after start)
PHASES = [('static', 0, 180), ('sin', 180, 216), ('rect', 216, 252), ('rest', 252, 1e9)]

if not os.path.exists(LOG):
    from google.colab import files; files.upload()
raw = pd.read_csv(LOG)

# time base: gyro timestamp t_us if it is valid, otherwise t_ms from time()
tu = raw.t_us.values.astype(float)
if np.all(np.diff(tu) > 0) and np.ptp(tu) > 1e6:
    t = (tu - tu[0]) / 1e6; tbase = 't_us (gyro timestamp)'
else:
    t = (raw.t_ms.values - raw.t_ms.values[0]) / 1e3; tbase = 't_ms (time())'
t += (raw.t_ms.values[0] / 1e3 - WARMUP_S)              # t = 0 at the start of the static phase
gz = raw.gz_cal.values / 1000.0                            # deg/s
theta_enc = r / L * (raw.encR.values - raw.encL.values)    # deg (encoders are in degrees)
phase = np.full(len(t), 'rest', dtype=object)
for name, a, b in PHASES: phase[(t >= a) & (t < b)] = name
log = pd.DataFrame(dict(t=t, gz=gz, theta_enc=theta_enc, phase=phase))
dt = np.diff(t); fs = 1 / np.median(dt)
raw.assign(phase=phase).to_csv('lab9_gyro_labeled.csv', index=False)   # same log with the phase label column
print(f'{len(log)} samples, time base: {tbase}, median dt {np.median(dt)*1000:.0f} ms (max {dt.max()*1000:.0f} ms), fs = {fs:.1f} Hz')
print(log.groupby('phase', sort=False).t.agg(['count', 'min', 'max']).round(1))

def integrate(t, w):
    return np.concatenate([[0], np.cumsum(0.5 * (w[1:] + w[:-1]) * np.diff(t))])
S = log[log.phase == 'static']; ts, ws = S.t.values - S.t.values[0], S.gz.values

## 1. Offset $b_0$
Baseline: mean of the first 30 s, standard error $\sigma/\sqrt{N}$, and the curve $b_0(T)$.

In [ ]:
m30 = ts <= 30
b0 = ws[m30].mean(); se = ws[m30].std(ddof=1) / np.sqrt(m30.sum())
print(f'b0 (first 30 s) = {b0:.4f} ± {se:.4f} deg/s   (full 3 min: {ws.mean():.4f})')
Ts = np.array([1, 2, 5, 10, 20, 30, 60, 120, 180])
bT = np.array([ws[ts <= T].mean() for T in Ts]); seT = np.array([ws[ts <= T].std(ddof=1) / np.sqrt((ts <= T).sum()) for T in Ts])
print(pd.DataFrame({'T [s]': Ts, 'b0(T)': bT.round(4), 'std err': seT.round(4)}).to_string(index=False))
plt.errorbar(Ts, bT, yerr=seT, marker='o', capsize=3); plt.axhline(ws.mean(), ls='--', c='gray', label='3-min mean')
plt.xscale('log'); plt.xlabel('calibration time T [s]'); plt.ylabel('b0(T) [deg/s]'); plt.title('Offset vs. calibration time'); plt.legend(); plt.show()

**My calibration method — `calibrate_gyro(log)`.** Median-based outlier rejection over the first `T_cal` seconds of the static log: take the median, drop samples further than 4·MAD from it (spikes, a bump on the table), then average what is left. The median makes it robust to spikes, the mean of the remaining samples keeps the precision of a plain average. `T_cal` = 20 s: the $b_0(T)$ curve is flat from about 10–20 s on, and longer only adds waiting time while the bias itself starts to drift (§2).

In [ ]:
def calibrate_gyro(log, T_cal=20.0, k=4.0):
    # Offset b0 [deg/s] from the static part of the log: robust mean of the first T_cal seconds.
    s = log[log.phase == 'static']
    w = s.gz.values[(s.t.values - s.t.values[0]) <= T_cal]
    med = np.median(w); mad = 1.4826 * np.median(np.abs(w - med))
    keep = np.abs(w - med) <= k * mad if mad > 0 else np.ones_like(w, bool)
    return w[keep].mean()

b0_cal = calibrate_gyro(log)
print(f'calibrate_gyro: b0 = {b0_cal:.4f} deg/s  (baseline 30-s mean {b0:.4f})')

R = log[log.phase != 'static'].reset_index(drop=True)
tr = R.t.values
th_gyro = integrate(tr, R.gz.values - b0_cal)
th_enc = R.theta_enc.values - R.theta_enc.values[0]
rest = R.phase.values == 'rest'
print(f'end of run (rest): theta_gyro = {th_gyro[rest].mean():+.2f} deg, theta_enc = {th_enc[rest].mean():+.2f} deg')
th_static = integrate(ts, ws - b0_cal)
print(f'heading error left after 3 min at rest with my b0: {th_static[-1]:+.2f} deg')

## 2. Bias drift $d$

In [ ]:
edges = np.arange(0, 181, 30); win_t = (edges[:-1] + edges[1:]) / 2
win_m = np.array([ws[(ts >= a) & (ts < b)].mean() for a, b in zip(edges[:-1], edges[1:])])
d_s, c0 = np.polyfit(win_t, win_m, 1); d = d_s * 60
th_raw = integrate(ts, ws); th_cor = integrate(ts, ws - b0)
slope_raw = np.polyfit(ts, th_raw, 1)[0] * 60
print('30-s means:', win_m.round(4)); print(f'd = {d:.4f} deg/s per min')
print(f'(a) uncorrected heading drift = {slope_raw:.2f} deg/min, b0*60 = {b0*60:.2f} deg/min')
for mnt in (1, 2, 3): print(f'(b) residual error after {mnt} min: {th_cor[np.searchsorted(ts, 60*mnt) - 1]:+.2f} deg')
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ts, ws, lw=0.3, alpha=0.4); ax[0].plot(win_t, win_m, 'o-'); ax[0].plot(win_t, np.polyval([d_s, c0], win_t), '--', label=f'd = {d:.3f} deg/s/min')
ax[0].set_ylim(b0 - 0.3, b0 + 0.3); ax[0].set_xlabel('t [s]'); ax[0].set_ylabel('deg/s'); ax[0].legend(); ax[0].set_title('Bias drift')
ax[1].plot(ts, th_raw, label='(a) raw'); ax[1].plot(ts, th_cor, label='(b) minus b0'); ax[1].set_xlabel('t [s]'); ax[1].set_ylabel('deg'); ax[1].legend(); ax[1].set_title('Heading at rest')
plt.tight_layout(); plt.show()

## 3. Gyro vs. encoder (Step 3) and measurement range $\pm\omega_{max}$

In [ ]:
w_enc = np.gradient(th_enc, tr)                             # encoder rate d(theta_enc)/dt
sign = np.sign(np.corrcoef(R.gz.values, w_enc)[0, 1])
ratio = np.polyfit(w_enc[np.abs(R.gz.values) < 200], (R.gz.values - b0_cal)[np.abs(R.gz.values) < 200], 1)[0]
print(f'encoder units check: gyro rate / encoder rate = {ratio:.3f} ->', 'encoders are in DEGREES' if 0.8 < abs(ratio) < 1.25 else 'encoders are NOT degrees (ticks?) - convert encL/encR before use')
print('sign check: gyro and encoder rate', 'AGREE (CCW positive for both)' if sign > 0 else 'DISAGREE -> check encoder sign / ports')
e = th_gyro - th_enc
fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
ax[0].plot(tr, th_gyro, label='θ_gyro'); ax[0].plot(tr, th_enc, '--', label='θ_enc'); ax[0].set_ylabel('deg'); ax[0].legend(); ax[0].set_title('Heading: gyro vs. encoders')
ax[1].plot(tr, e, c='C3'); ax[1].set_ylabel('e = θ_gyro − θ_enc [deg]')
ax[2].plot(tr, w_enc, lw=0.8, label='dθ_enc/dt'); ax[2].plot(tr, R.gz.values - b0_cal, lw=0.8, label='ω_z gyro'); ax[2].set_ylabel('deg/s'); ax[2].set_xlabel('t [s]'); ax[2].legend()
for a in ax:
    for name, a0, a1 in PHASES[1:3]: a.axvspan(a0, a1, alpha=0.06, color='k')
plt.tight_layout(); plt.show()

wmax_pos, wmax_neg = R.gz.max(), R.gz.min()
print(f'+omega_max = {wmax_pos:.2f} deg/s ({(R.gz == wmax_pos).sum()} samples), -omega_max = {wmax_neg:.2f} deg/s ({(R.gz == wmax_neg).sum()} samples)')
print(f'fastest encoder rate: {w_enc.max():.0f} / {w_enc.min():.0f} deg/s')
plt.figure(figsize=(6, 6)); plt.plot(w_enc, R.gz.values - b0_cal, '.', ms=2)
lim = np.abs(w_enc).max() * 1.05; plt.plot([-lim, lim], [-lim, lim], 'k--', lw=0.8, label='ideal')
plt.axhline(wmax_pos, c='r', ls=':'); plt.axhline(wmax_neg, c='r', ls=':', label='saturation')
plt.xlabel('encoder rate [deg/s]'); plt.ylabel('gyro rate [deg/s]'); plt.title('Flat top = saturation'); plt.legend(); plt.show()

## 4. Resolution $\Delta$

In [ ]:
vals = np.unique(S.gz.values * 1000); gaps = np.diff(vals)
print('distinct static values:', len(vals), ' gap counts:', pd.Series(gaps.round(0)).value_counts().sort_index().to_dict())
delta = gaps[gaps < 15].mean() / 1000
print(f'Delta = {delta*1000:.2f} mdeg/s = {delta:.5f} deg/s   (check: omega_max / 32767 = {wmax_pos/32767*1000:.2f} mdeg/s)')

## 5. Noise $\sigma$

In [ ]:
en = ws - ws.mean(); sigma = en.std(ddof=1)
print(f'sigma = {sigma:.4f} deg/s at {fs:.1f} Hz -> R = {sigma**2:.5f} (deg/s)^2;  skew {stats.skew(en):.3f}, excess kurtosis {stats.kurtosis(en):.3f}, beyond 4 sigma: {(np.abs(en) > 4*sigma).sum()}')
x = np.linspace(en.min(), en.max(), 300)
plt.hist(en, bins=np.arange(en.min() - delta/2, en.max() + delta, delta), density=True, alpha=0.6, label='measured')
plt.plot(x, stats.norm.pdf(x, 0, sigma), 'r', label=f'Gaussian σ = {sigma:.3f}'); plt.xlabel('rate − b0 [deg/s]'); plt.legend(); plt.title('Noise histogram'); plt.show()

## 6. Scale factor $s$ (CW / CCW, per amplitude)
The run is cut into half-periods (one turn direction each). Saturated half-periods (gyro at its limit) are excluded from the fit and shown separately.

In [ ]:
rows = []
for name, a0, a1 in PHASES[1:3]:
    for j in range(int((a1 - a0) / 2)):                     # half-periods of 2 s
        m = (tr >= a0 + 2*j) & (tr < a0 + 2*j + 2)
        if m.sum() < 5: continue
        i0, i1 = np.where(m)[0][[0, -1]]
        dg, de = th_gyro[i1] - th_gyro[i0], th_enc[i1] - th_enc[i0]
        if abs(de) < 5: continue
        sat = (np.abs(R.gz.values[i0:i1+1]) >= 0.99 * max(wmax_pos, -wmax_neg)).any()
        rows.append(dict(signal=name, A=[30, 60, 150][min(j // 6, 2)], direction='CCW' if de > 0 else 'CW', d_enc=de, d_gyro=dg, saturated=sat, s_pct=(dg / de - 1) * 100))
seg = pd.DataFrame(rows)
print(seg.groupby(['signal', 'A', 'direction', 'saturated']).s_pct.agg(['count', 'mean']).round(2))
ok = seg[~seg.saturated]
fit = {}
for dirn in ('CCW', 'CW'):
    g = ok[ok.direction == dirn]; slope = np.polyfit(g.d_enc, g.d_gyro, 1)[0]; fit[dirn] = (slope - 1) * 100
s_ccw, s_cw = fit['CCW'], fit['CW']; s_all = (s_ccw + s_cw) / 2
print(f'\ns CCW = {s_ccw:+.2f} %,  s CW = {s_cw:+.2f} %  (slope of Δθ_gyro vs Δθ_enc, unsaturated half-periods)')
ns = ~(np.abs(R.gz.values) >= 0.99 * max(wmax_pos, -wmax_neg))
plt.figure(figsize=(6, 6))
for dirn, c in (('CCW', 'C0'), ('CW', 'C1')):
    g = seg[seg.direction == dirn]; plt.scatter(g.d_enc, g.d_gyro, c=np.where(g.saturated, 'r', c), s=18, label=dirn)
lim = np.abs(seg.d_enc).max() * 1.1; plt.plot([-lim, lim], [-lim, lim], 'k--', lw=0.8)
plt.xlabel('Δθ_enc per half-period [deg]'); plt.ylabel('Δθ_gyro [deg]'); plt.title('Scale-factor fit (red = saturated)'); plt.legend(); plt.show()
plt.figure(figsize=(6, 6)); plt.plot(th_enc, th_gyro, '.', ms=2, label='rotation run')
p = np.polyfit(th_enc[ns], th_gyro[ns], 1); xx = np.array([th_enc.min(), th_enc.max()])
plt.plot(xx, np.polyval(p, xx), 'r', label=f'fit (unsaturated): slope 1+s = {p[0]:.4f}')
plt.xlabel('θ_enc [deg]'); plt.ylabel('θ_gyro [deg]'); plt.title('θ_gyro vs θ_enc'); plt.legend(); plt.show()
print(f'slip check: tape mark says {SLIP_DEG:+.1f} deg at the end; gyro {th_gyro[rest].mean():+.1f}, encoders {th_enc[rest].mean():+.1f} deg')
print('Raw -> deg/s: readRawData() is not available in block diagrams, so gz_cal is used; factor = Delta =', round(delta, 5), 'deg/s per LSB')

## 7. Simulated gyro `gyro_sim(omega_true, t)`
Fed with the encoder rate of the rotation run as the true rotation, overlaid on the real gyro.

In [ ]:
def gyro_sim(omega_true, t, b0=b0, d=d / 60, s=s_all / 100, wmax=(wmax_neg, wmax_pos), delta=delta, sigma=sigma, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    t = np.asarray(t, float)
    w = (1 + s) * np.asarray(omega_true, float) + b0 + d * (t - t[0]) + rng.normal(0, sigma, len(t))
    return np.round(np.clip(w, wmax[0], wmax[1]) / delta) * delta

w_sim = gyro_sim(w_enc, tr, b0=b0_cal, d=0, rng=np.random.default_rng(0))
plt.plot(tr, R.gz.values, lw=0.8, label='real gyro'); plt.plot(tr, w_sim, lw=0.8, alpha=0.8, label='gyro_sim(encoder rate)')
plt.xlabel('t [s]'); plt.ylabel('deg/s'); plt.title('Real vs. simulated gyro'); plt.legend(); plt.show()

## 8. Parameter table

In [ ]:
table = pd.DataFrame([
    ['Offset (bias)', 'b0', f'{b0:.3f} ± {se:.3f} (calibrate_gyro: {b0_cal:.3f})', 'deg/s', 'static mean, T = 30 s'],
    ['Bias drift', 'd', f'{d:.4f}', 'deg/s/min', '30-s window means'],
    ['Heading drift, uncorrected', '', f'{slope_raw:.1f}', 'deg/min', 'slope of θ(t)'],
    ['Measurement range', '±ω_max', f'+{wmax_pos:.1f} / {wmax_neg:.1f}', 'deg/s', 'saturation in rotation run'],
    ['Resolution', 'Δ', f'{delta:.5f}', 'deg/s', 'quantisation step'],
    ['Noise std', 'σ', f'{sigma:.4f}', 'deg/s', f'static, 3 min, {fs:.0f} Hz'],
    ['Raw → deg/s factor', '', f'{delta:.5f}', 'deg/s per LSB', 'Δ (readRawData not available in blocks)'],
    ['Scale-factor error CCW / CW', 's', f'{s_ccw:+.2f} / {s_cw:+.2f}', '%', 'gyro vs. encoder angle'],
    ['Positive rotation direction', '', 'CCW' if sign > 0 else 'CW', '—', '§6'],
], columns=['Parameter', 'Symbol', 'Value', 'Unit', 'Method'])
table['Log'] = ['lab9_gyro.csv: ' + p for p in ['static', 'static', 'static', 'sin + rect', 'static', 'static', 'static', 'sin + rect', 'sin + rect']]
print(table.to_string(index=False)); table.to_csv('lab9_parameters.csv', index=False)

## 9. Error budget and short analysis

In [ ]:
res = abs(th_static[-1]) / 180                              # residual rate left by calibrate_gyro [deg/s]
bud = pd.DataFrame([dict(t=f'{T} s', offset_uncorrected=abs(b0)*T, after_my_calibration=res*T, drift=0.5*abs(d/60)*T**2,
                         noise=sigma*np.sqrt(T/fs), scale_per_turn=abs(s_all)/100*360) for T in (10, 60, 180)]).set_index('t').round(2)
print('heading error [deg]'); print(bud.to_string())
dom = lambda T: bud.loc[f'{T} s', ['after_my_calibration', 'drift', 'noise']].astype(float).idxmax()
print(f'''
Without calibration the offset dominates: {abs(b0)*10:.1f} deg after 10 s, {abs(b0)*60:.0f} deg after 1 min, {abs(b0)*180:.0f} deg after 3 min.
With calibrate_gyro (robust mean, 20 s) the error left at rest is {th_static[-1]:+.2f} deg after 3 min.
After calibration the largest remaining term is: 10 s -> {dom(10)}, 1 min -> {dom(60)}, 3 min -> {dom(180)};
while turning, the scale factor adds about {abs(s_all)/100*360:.1f} deg per full turn.
In e(t): steady growth at rest = bias, a step at every turn = scale factor, steps only on the fastest turns = saturation,
jumps at hard accelerations (rect edges) = wheel slip in the encoders; the tape mark ({SLIP_DEG:+.1f} deg) shows which sensor is closer.''')